# M1.0a Primer: Vectors and Fields

Book pointer: chapter 1, sections 1.1 (vectors) and 1.2 (multivariate functions and vector fields).

This is the first of four primer notebooks (00a to 00d) that teach the mathematics of chapter 1 from the ground up. It is written to be read on its own: everything needed to follow it, and to follow the module 1 notebooks that come after it, is in the text. Each idea is built from a picture or a measurement before the formula is written, the algebra is done by hand in worked examples, and every number is then reproduced in plain NumPy so nothing is hidden. Section 2 is the lesson in prose. Section 3 is the same lesson with the sums and products done explicitly. Sections 6 and 7 are a formula sheet and the answers to the warm-up questions.

If the warm-up questions are easy, skip to `01_grids_and_operators`, which is the compact version of all four primers.

## 1. Warm-up

Try on paper first. Answers are in section 7.

1. What is the difference between a vector and its components? Why can the same vector have different components?
2. $\mathbf{a}\cdot\mathbf{b}$ and $\mathbf{a}\times\mathbf{b}$: which is a number, which is a vector, and what does each measure?
3. What is the unit vector in the direction of $(3, 4, 0)$?
4. A scalar field and a vector field: give one physical example of each and say what a picture of each looks like.
5. In Coulomb's law the vector from the source charge to the observation point appears. Write it in terms of the two position vectors.
6. Two contour lines of a scalar field can never cross. Why not?
7. A field-line picture shows lines bunching together in one region. What does that say about the field there?


## 2. The ideas

**Why vectors.** A temperature is one number. A force is not: pushing north and pushing east are different, and so are pushing hard and pushing gently. A quantity with a size and a direction is a vector, and electromagnetism is full of them: forces, velocities, the electric field, the magnetic field. The rules for combining vectors (adding them, scaling them, multiplying them in two different ways) are the grammar of the subject, and section 1.1 of the book is where that grammar is laid down. The whole of this notebook is that grammar plus one more idea, the field.

**A vector is an arrow, not a list of numbers.** This is the first thing to get straight. The arrow exists on its own: it has a length and it points somewhere. To calculate with it we pick three perpendicular reference directions and record how far the arrow goes along each. Those three numbers are the components. They are a description of the arrow relative to the axes we chose, and a different choice of axes gives different components for the same arrow. The length of the arrow and the angle between two arrows do not depend on the axes, which is why formulas built from them (the dot and cross products below) describe physics and formulas built from a single component usually do not.

**The two products answer two different questions.** Given two vectors, the dot product asks: how much of one lies along the other? It returns a number, largest when the two are parallel, zero when they are perpendicular, negative when they oppose. Physics uses it whenever only the aligned part matters: the work done by a force is the part of the force along the displacement, and the flux of a field through a surface is the part of the field along the surface normal. The cross product asks a different question: what is the axis perpendicular to both, and how large is the parallelogram they span? It returns a vector. Physics uses it when a result is perpendicular to two inputs: the magnetic force on a moving charge, the torque of a force about a pivot, and the surface element of a curved surface, which is built from two tangent vectors.

**Position, and the vector that matters most.** A point in space is labelled by its position vector $\mathbf{r}$, the arrow from an agreed origin to the point. Almost every formula in electrostatics contains not a position but a difference of two positions: the arrow from the place where a charge sits to the place where we want to know the field. Coulomb's law says the field is along that arrow and falls off as the inverse square of its length. Getting that arrow right (which end is which, and dividing by the cube of its length rather than the square, because the arrow itself carries one factor of length) is half of every superposition calculation in module 1.

**Fields.** A function of one variable is a curve. A function of position, $f(x, y, z)$, is a field: a value attached to every point of space. If the value is a number it is a scalar field (temperature, pressure, the electric potential). If the value is a vector it is a vector field (wind velocity, the electric and magnetic fields). Fields are the central objects of the course. The laws of electromagnetism are statements about how fields vary from point to point, and the operators of notebook 00c (gradient, divergence, curl) are the tools that measure that variation.

**Pictures of fields.** A scalar field in two dimensions can be drawn as a landscape, height standing for the value, or as a contour map, curves along which the value is constant. The contour map is the more useful picture and it has reading rules: closely spaced contours mean the field changes quickly, closed loops surround a maximum or a minimum, and two contours can never cross because a point cannot have two values. In three dimensions the contours become level surfaces, shells on which the value is constant. A vector field is drawn either as a sample of arrows (a quiver plot) or as field lines, curves that follow the arrows everywhere (a stream plot). Field lines of the electric field obey rules of their own: they start on positive charge, end on negative charge or at infinity, never cross, and crowd together where the field is strong.

**Fields on a computer.** We cannot store a value at every point, so we store values on a grid: a list of $x$ positions, a list of $y$ positions, and a value at every combination. A scalar field becomes a two-dimensional array, a vector field becomes an array with one extra axis holding the components, and every operation of the course (sums over charges, derivatives, integrals) becomes an array operation. The one convention to fix now is which array index follows which coordinate. These notebooks always use the index order $[i, j]$ for $(x_i, y_j)$, which is `np.meshgrid(..., indexing="ij")`. The default mode of `meshgrid` swaps the axes, and a field built in one convention and plotted in the other comes out transposed with no error message.

**Where this goes.** Notebook 01 of module 1 uses these grids to compute gradient, divergence and curl. Notebook 02 builds the electric field of many charges by adding Coulomb arrows, which is section 3.4 and 3.5 of this primer repeated a million times. Notebook 03 builds the potential, a scalar field, and recovers the electric field from its contour map. Every one of those steps is a vector or field operation from this notebook.


## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True)

### 3.1 What a vector is (section 1.1)

A vector has a length and a direction. That is the geometric object. To calculate with it we choose three perpendicular reference directions $\hat{\mathbf{x}}, \hat{\mathbf{y}}, \hat{\mathbf{z}}$ (unit vectors, length one) and write the vector as a sum of multiples of them:

$$\mathbf{a} = a_x\hat{\mathbf{x}} + a_y\hat{\mathbf{y}} + a_z\hat{\mathbf{z}}, \qquad \text{written for short as } (a_x, a_y, a_z).$$

The three numbers are the components. They depend on the reference directions: rotate the axes and the same arrow gets different components. The length does not depend on the axes:

$$|\mathbf{a}| = \sqrt{a_x^2 + a_y^2 + a_z^2}.$$

This is Pythagoras applied twice: first in the $xy$ plane to get the length of the shadow, $\sqrt{a_x^2 + a_y^2}$, then once more with the $z$ component. A vector with length zero is the zero vector and has no direction.

Two operations are built in. Adding vectors adds components (geometrically, place them head to tail, or complete the parallelogram; the two pictures give the same arrow). Multiplying by a number scales the length (and flips the direction if the number is negative). Subtraction is adding the negative: $\mathbf{a} - \mathbf{b}$ is the arrow from the head of $\mathbf{b}$ to the head of $\mathbf{a}$ when both start at the same point. Dividing a vector by its own length gives the unit vector $\hat{\mathbf{a}} = \mathbf{a}/|\mathbf{a}|$, which keeps the direction and throws away the length. Unit vectors are how a direction is handed to a formula: "a force of 5 N in the direction of $\mathbf{a}$" is $5\hat{\mathbf{a}}$.

**Worked example 1.** $\mathbf{a} = (3, 4, 0)$. Length $|\mathbf{a}| = \sqrt{9 + 16} = 5$. Unit vector $\hat{\mathbf{a}} = (3, 4, 0)/5 = (0.6, 0.8, 0)$, and indeed $0.6^2 + 0.8^2 = 1$. Adding $\mathbf{b} = (1, -1, 2)$ gives $\mathbf{a} + \mathbf{b} = (4, 3, 2)$. Scaling by $-2$ gives $-2\mathbf{a} = (-6, -8, 0)$, the same line, opposite direction, twice as long.

**Worked example 2 (same arrow, new axes).** Keep $\mathbf{a} = (3, 4, 0)$ but rotate the $x$ and $y$ axes counter-clockwise by $30^\circ$. The new axis directions, written in the old components, are $\hat{\mathbf{x}}' = (\cos30^\circ, \sin30^\circ, 0) = (0.866, 0.5, 0)$ and $\hat{\mathbf{y}}' = (-0.5, 0.866, 0)$. The new components are the lengths of the shadows of $\mathbf{a}$ on the new axes (the dot products of section 3.2): $a_x' = 3(0.866) + 4(0.5) = 4.598$ and $a_y' = 3(-0.5) + 4(0.866) = 1.964$. Different numbers, same arrow: $\sqrt{4.598^2 + 1.964^2} = \sqrt{21.14 + 3.86} = 5$. Any quantity that physics cares about must come out the same in both descriptions, which is the test every vector formula has to pass.

**Worked example 3 (adding forces).** Coulomb's law, met properly in notebook 00e and chapter 2, says the force on a charge from another charge is along the line joining them, repulsive for like charges, with a size proportional to $1/(\text{distance})^2$. Put a positive charge at the origin and two equal positive charges at $(2, 0, 0)$ and $(0, 2, 0)$. In units where the force between unit charges at unit distance is 1, the first charge pushes the origin charge away from itself with force $(-2, 0, 0)/2^3 = (-0.25, 0, 0)$ (the arrow from the source to the origin, divided by the cube of its length, as section 3.4 explains). The second gives $(0, -0.25, 0)$. The total force is the vector sum $(-0.25, -0.25, 0)$, of size $0.354$, pointing at $225^\circ$, straight away from the pair. Forces from several charges add as vectors, never as numbers; this is the superposition principle and it is the whole content of notebook 02.


In [ ]:
a = np.array([3.0, 4.0, 0.0])
b = np.array([1.0, -1.0, 2.0])

length = np.linalg.norm(a)                    # sqrt(a @ a)
a_hat = a / length
print("length of a:", length, "   unit vector:", a_hat, "   its length:", np.linalg.norm(a_hat))
print("a + b =", a + b, "    -2 a =", -2 * a)

# A picture of head-to-tail addition in the plane.
fig, ax = plt.subplots(figsize=(5.5, 5))
def arrow(start, vec, colour, label):
    ax.annotate("", xy=(start + vec)[:2], xytext=start[:2], arrowprops=dict(arrowstyle="->", color=colour, lw=2))
    ax.text(*(start + vec / 2)[:2] + np.array([0.1, 0.1]), label, color=colour)
arrow(np.zeros(3), a, "C0", "a")
arrow(a, b, "C1", "b (placed at the head of a)")
arrow(np.zeros(3), a + b, "C2", "a + b")
ax.set_xlim(-0.5, 5); ax.set_ylim(-0.5, 5); ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title("vector addition is head to tail")
plt.show()

In [ ]:
# Worked example 2: the same arrow in rotated axes.
a = np.array([3.0, 4.0, 0.0])
th = np.radians(30)
x_new = np.array([np.cos(th), np.sin(th), 0.0]); y_new = np.array([-np.sin(th), np.cos(th), 0.0])
a_new = np.array([a @ x_new, a @ y_new, a[2]])
print("components in the old axes:", a, "   in axes rotated by 30 degrees:", a_new)
print("length in old axes:", np.linalg.norm(a), "   length in new axes:", np.linalg.norm(a_new))

# Worked example 3: forces on a charge at the origin from two charges, added as vectors.
sources = np.array([[2.0, 0.0, 0.0], [0.0, 2.0, 0.0]])
total = np.zeros(3)
for s in sources:
    R = np.zeros(3) - s                                 # from the source to the origin
    F = R / np.linalg.norm(R) ** 3
    total += F
    print(f"force from the charge at {s}: {F}")
print(f"total force: {total}   size {np.linalg.norm(total):.3f}   direction {np.degrees(np.arctan2(total[1], total[0])) % 360:.0f} degrees")


### 3.2 The dot product

The dot product turns two vectors into a number. It has three equivalent faces, and the skill is to switch between them freely.

1. Components: $\mathbf{a}\cdot\mathbf{b} = a_x b_x + a_y b_y + a_z b_z$. This is how you compute it.
2. Geometry: $\mathbf{a}\cdot\mathbf{b} = |\mathbf{a}||\mathbf{b}|\cos\theta$, with $\theta$ the angle between them. This is how you interpret it: it is zero when they are perpendicular, largest when parallel, negative when they point against each other.
3. Projection: $\mathbf{a}\cdot\hat{\mathbf{b}}$ is the length of the shadow of $\mathbf{a}$ on the line of $\mathbf{b}$. So $(\mathbf{a}\cdot\hat{\mathbf{b}})\,\hat{\mathbf{b}}$ is the part of $\mathbf{a}$ along $\mathbf{b}$, and what is left over, $\mathbf{a} - (\mathbf{a}\cdot\hat{\mathbf{b}})\hat{\mathbf{b}}$, is perpendicular to $\mathbf{b}$.

Why faces 1 and 2 agree is worth seeing once. The unit vectors satisfy $\hat{\mathbf{x}}\cdot\hat{\mathbf{x}} = 1$ (parallel, $\cos0 = 1$) and $\hat{\mathbf{x}}\cdot\hat{\mathbf{y}} = 0$ (perpendicular). Expand $(a_x\hat{\mathbf{x}} + a_y\hat{\mathbf{y}} + a_z\hat{\mathbf{z}})\cdot(b_x\hat{\mathbf{x}} + \dots)$ term by term: nine products, six of them zero, and face 1 is what remains. Since face 2 is a statement about lengths and an angle, which do not depend on the axes, the component formula gives the same number in every coordinate system, which worked example 2 already hinted at.

Properties worth knowing by heart: it is symmetric, $\mathbf{a}\cdot\mathbf{b} = \mathbf{b}\cdot\mathbf{a}$; it distributes over addition, $\mathbf{a}\cdot(\mathbf{b} + \mathbf{c}) = \mathbf{a}\cdot\mathbf{b} + \mathbf{a}\cdot\mathbf{c}$; and $\mathbf{a}\cdot\mathbf{a} = |\mathbf{a}|^2$, so a length is the square root of a dot product. Because $|\cos\theta| \le 1$, the dot product can never exceed the product of the lengths: $|\mathbf{a}\cdot\mathbf{b}| \le |\mathbf{a}||\mathbf{b}|$, with equality only for parallel vectors.

Physics uses the dot product whenever only the part of one vector along another matters. Work is $\mathbf{F}\cdot\mathbf{d}$, force along displacement: pushing sideways on a moving cart does no work. Flux is $\mathbf{E}\cdot\hat{\mathbf{n}}\,A$, field along the surface normal times area: a field that runs parallel to a surface passes nothing through it. Both are everywhere in this course.

**Worked example 4.** $\mathbf{a} = (3, 1, 0)$, $\mathbf{b} = (1, 2, 0)$. Face 1: $\mathbf{a}\cdot\mathbf{b} = 3 + 2 = 5$. Lengths $\sqrt{10}$ and $\sqrt5$. Face 2: $\cos\theta = 5/\sqrt{50} = 1/\sqrt2$, so $\theta = 45^\circ$. Face 3: $\hat{\mathbf{b}} = (1, 2, 0)/\sqrt5$, shadow length $\mathbf{a}\cdot\hat{\mathbf{b}} = 5/\sqrt5 = \sqrt5 = 2.236$, part of $\mathbf{a}$ along $\mathbf{b}$ is $\sqrt5\,\hat{\mathbf{b}} = (1, 2, 0)$, and the perpendicular remainder is $(3, 1, 0) - (1, 2, 0) = (2, -1, 0)$. Check: $(2, -1, 0)\cdot(1, 2, 0) = 0$.

**Worked example 5.** Is $(1, 2, 3)$ perpendicular to $(3, 0, -1)$? Dot product $3 + 0 - 3 = 0$. Yes. What angle does $(1, 1, 1)$ make with the $z$ axis? $\cos\theta = (1,1,1)\cdot(0,0,1)/(\sqrt3\cdot1) = 1/\sqrt3$, $\theta = 54.7^\circ$.

**Worked example 6 (distances).** The distance between two points is the length of their difference, and a length is the square root of a dot product. $P = (1, 2, 3)$ and $Q = (4, 6, 3)$: $\mathbf{d} = Q - P = (3, 4, 0)$, $\mathbf{d}\cdot\mathbf{d} = 25$, distance $5$. Expanding $(\mathbf{a} - \mathbf{b})\cdot(\mathbf{a} - \mathbf{b})$ gives $|\mathbf{a}|^2 + |\mathbf{b}|^2 - 2\mathbf{a}\cdot\mathbf{b}$, which is the law of cosines; for the vectors of worked example 4, $|\mathbf{a} - \mathbf{b}|^2 = |(2, -1, 0)|^2 = 5$ and $10 + 5 - 2\cdot5 = 5$.

**Worked example 7 (flux through a flat surface).** A uniform field $\mathbf{E} = (0, 0, 3)$ V/m crosses a flat plate of area $A = 2\ \mathrm{m^2}$ whose unit normal is tilted $30^\circ$ from the $z$ axis, $\hat{\mathbf{n}} = (0, \sin30^\circ, \cos30^\circ) = (0, 0.5, 0.866)$. The flux is $\mathbf{E}\cdot\hat{\mathbf{n}}\,A = 3\cdot0.866\cdot2 = 5.196$ V m. Tilt the plate to $90^\circ$ so $\hat{\mathbf{n}} = (0, 1, 0)$: flux $3\cdot0\cdot2 = 0$, the field slides past the plate. Face-on, flux $= 6$ V m. Only the component of $\mathbf{E}$ along the normal counts, and $\cos\theta$ is how the dot product says so. Notebook 00b turns this into an integral over curved surfaces, and chapter 4 turns it into Gauss' law.


**Animation.** $\mathbf{a} = (3, 1, 0)$ stays fixed while $\mathbf{b}$, of length $\sqrt5$, turns a full circle. It starts at $(1, 2, 0)$, the moment of worked example 2: angle $45^\circ$, $\mathbf{a}\cdot\mathbf{b} = 5$, shadow $\sqrt5 = 2.24$. Watch the shadow of $\mathbf{a}$ on the dashed line of $\mathbf{b}$: it shrinks to nothing at $90^\circ$, then reappears on the far side, pointing against $\mathbf{b}$, and the dot product turns negative. The three faces of the dot product are the three readouts.

In [ ]:
from emlib import show
show.animation("m1_fields", "DotProductShadow")

In [ ]:
a = np.array([3.0, 1.0, 0.0]); b = np.array([1.0, 2.0, 0.0])
dot = a @ b
theta = np.degrees(np.arccos(dot / (np.linalg.norm(a) * np.linalg.norm(b))))
b_hat = b / np.linalg.norm(b)
shadow = a @ b_hat
a_along = shadow * b_hat
a_perp = a - a_along
print(f"a . b = {dot}   angle = {theta:.1f} deg   shadow of a on b = {shadow:.3f}")
print(f"part of a along b = {a_along}   perpendicular remainder = {a_perp}   check remainder . b = {a_perp @ b}")
print(f"(1,2,3) . (3,0,-1) = {np.array([1, 2, 3]) @ np.array([3, 0, -1])}    angle of (1,1,1) with z axis = {np.degrees(np.arccos(1 / np.sqrt(3))):.1f} deg")

fig, ax = plt.subplots(figsize=(5, 5))
for v, c, lbl in ((a, "C0", "a"), (b, "C1", "b"), (a_along, "C2", "along b"), (a_perp, "C3", "perpendicular to b")):
    ax.annotate("", xy=v[:2], xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=c, lw=2)); ax.text(*v[:2] * 1.06, lbl, color=c)
ax.plot([a_along[0], a[0]], [a_along[1], a[1]], "k:", lw=1)
ax.set_xlim(-0.5, 3.5); ax.set_ylim(-1.5, 2.5); ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title("the dot product splits a into along-b and across-b")
plt.show()

In [ ]:
# Worked example 6: distance as the square root of a dot product, and the law of cosines.
P = np.array([1.0, 2.0, 3.0]); Qp = np.array([4.0, 6.0, 3.0])
d = Qp - P
print(f"d = {d}   d . d = {d @ d}   distance = {np.sqrt(d @ d)}")
a = np.array([3.0, 1.0, 0.0]); b = np.array([1.0, 2.0, 0.0])
print(f"|a - b|^2 = {(a - b) @ (a - b)}   |a|^2 + |b|^2 - 2 a.b = {a @ a + b @ b - 2 * (a @ b)}")
print(f"|a . b| = {abs(a @ b)} <= |a||b| = {np.linalg.norm(a) * np.linalg.norm(b):.3f}")

# Worked example 7: flux of a uniform field through a tilted flat plate.
E = np.array([0.0, 0.0, 3.0]); A = 2.0
for tilt in (0.0, 30.0, 60.0, 90.0):
    n_hat = np.array([0.0, np.sin(np.radians(tilt)), np.cos(np.radians(tilt))])
    print(f"plate tilted {tilt:4.0f} degrees: flux = E . n A = {E @ n_hat * A:.3f} V m")


### 3.3 The cross product

The cross product turns two vectors into a third vector that is perpendicular to both. Its length is the area of the parallelogram the two span, $|\mathbf{a}\times\mathbf{b}| = |\mathbf{a}||\mathbf{b}|\sin\theta$, and its direction follows the right-hand rule.

The right-hand rule, stated carefully: hold your right hand flat with the fingers pointing along $\mathbf{a}$. Curl the fingers towards $\mathbf{b}$ through the smaller of the two angles between them. The thumb now points along $\mathbf{a}\times\mathbf{b}$. For $\mathbf{a} = \hat{\mathbf{x}}$ and $\mathbf{b} = \hat{\mathbf{y}}$ the fingers curl counter-clockwise when viewed from above and the thumb points up, $+\hat{\mathbf{z}}$. Swap the order and the fingers curl the other way, so $\hat{\mathbf{y}}\times\hat{\mathbf{x}} = -\hat{\mathbf{z}}$. The rule is a convention (left-handed axes would flip every sign), but it is the convention used everywhere in physics, and the coordinate axes themselves are chosen so that $\hat{\mathbf{x}}\times\hat{\mathbf{y}} = \hat{\mathbf{z}}$.

In components,
$$\mathbf{a}\times\mathbf{b} = (a_y b_z - a_z b_y)\,\hat{\mathbf{x}} + (a_z b_x - a_x b_z)\,\hat{\mathbf{y}} + (a_x b_y - a_y b_x)\,\hat{\mathbf{z}}.$$

A way to remember it: each component is built from the other two indices in cyclic order ($x \to yz$, $y \to zx$, $z \to xy$), as "first times second minus second times first". The determinant with $\hat{\mathbf{x}}, \hat{\mathbf{y}}, \hat{\mathbf{z}}$ in the top row, the components of $\mathbf{a}$ in the second and of $\mathbf{b}$ in the third gives the same thing.

Properties: it is antisymmetric, $\mathbf{b}\times\mathbf{a} = -\mathbf{a}\times\mathbf{b}$, so the order of the factors matters and carries the orientation; a vector crossed with itself (or with anything parallel to it) is zero, because the parallelogram collapses; it distributes over addition; and for the unit vectors $\hat{\mathbf{x}}\times\hat{\mathbf{y}} = \hat{\mathbf{z}}$, $\hat{\mathbf{y}}\times\hat{\mathbf{z}} = \hat{\mathbf{x}}$, $\hat{\mathbf{z}}\times\hat{\mathbf{x}} = \hat{\mathbf{y}}$ (cyclic), with a minus sign if the order is reversed. The antisymmetry is not a nuisance: when the cross product gives the normal to a surface, the order of the two tangent vectors decides which side of the surface the normal points to, and in notebook 00b "outward" is fixed by choosing that order.

Physics uses it whenever a result is perpendicular to two inputs. The magnetic force on a moving charge is $q\mathbf{v}\times\mathbf{B}$ (chapter 11): perpendicular to the velocity, so it never does work. Torque is $\mathbf{r}\times\mathbf{F}$: a force applied at $\mathbf{r}$ from a pivot turns the body about the axis perpendicular to both. The Biot-Savart law has $d\mathbf{l}\times\hat{\mathbf{r}}$ (chapter 12).

**Worked example 8.** $\mathbf{a} = (3, 1, 0)$, $\mathbf{b} = (1, 2, 0)$. $x$: $a_y b_z - a_z b_y = 0 - 0 = 0$. $y$: $a_z b_x - a_x b_z = 0 - 0 = 0$. $z$: $a_x b_y - a_y b_x = 6 - 1 = 5$. So $\mathbf{a}\times\mathbf{b} = (0, 0, 5)$: perpendicular to the plane of $\mathbf{a}$ and $\mathbf{b}$, pointing along $+z$ as the right hand predicts for a counter-clockwise turn from $\mathbf{a}$ to $\mathbf{b}$, with length $5 = \sqrt{10}\sqrt5\sin45^\circ$.

**Worked example 9.** $\mathbf{a} = (1, 2, 3)$, $\mathbf{b} = (4, 5, 6)$. $x$: $2\cdot6 - 3\cdot5 = -3$. $y$: $3\cdot4 - 1\cdot6 = 6$. $z$: $1\cdot5 - 2\cdot4 = -3$. $\mathbf{a}\times\mathbf{b} = (-3, 6, -3)$. Check perpendicularity: $(-3, 6, -3)\cdot(1, 2, 3) = -3 + 12 - 9 = 0$ and $\cdot(4, 5, 6) = -12 + 30 - 18 = 0$.

**Worked example 10 (volume).** The scalar triple product $\mathbf{a}\cdot(\mathbf{b}\times\mathbf{c})$ is the volume of the parallelepiped spanned by the three vectors (with a sign that records the handedness). For $\mathbf{a} = (1, 0, 0)$, $\mathbf{b} = (0, 2, 0)$, $\mathbf{c} = (0, 0, 3)$: $\mathbf{b}\times\mathbf{c} = (6, 0, 0)$ and $\mathbf{a}\cdot(6, 0, 0) = 6$, the volume of the $1\times2\times3$ box.

**Worked example 11 (area and torque).** A triangle has vertices $P = (0, 0, 0)$, $Q = (2, 0, 0)$, $R = (1, 3, 0)$. Two sides from $P$ are $(2, 0, 0)$ and $(1, 3, 0)$; their cross product is $(0, 0, 6)$, the parallelogram has area $6$, and the triangle is half of it: area $3$. Check: base $2$, height $3$, $\tfrac12\cdot2\cdot3 = 3$. Now torque. A force $\mathbf{F} = (0, 2, 0)$ N acts at $\mathbf{r} = (0.5, 0, 0)$ m from a pivot: $\boldsymbol{\tau} = \mathbf{r}\times\mathbf{F} = (0, 0, 1)$ N m, a turn about the $z$ axis, counter-clockwise seen from above. The same force applied at $\mathbf{r} = (0, 0.5, 0)$, along its own line of action, gives zero: a force through the pivot cannot turn anything. For $\mathbf{r} = (1, 2, 0)$ and $\mathbf{F} = (3, 1, 0)$ the $z$ component is $1\cdot1 - 2\cdot3 = -5$: clockwise.


**Animation.** The same $\mathbf{a} = (3, 1, 0)$ with $\mathbf{b}$ turning a full circle, now with the parallelogram the two span. Its area is $|\mathbf{a}\times\mathbf{b}| = |\mathbf{a}||\mathbf{b}|\sin\theta$: 5 at the start (worked example 4), largest at $90^\circ$, zero when the vectors line up. The symbol in the middle is the direction of $\mathbf{a}\times\mathbf{b}$: out of the page while $\mathbf{b}$ is counter-clockwise from $\mathbf{a}$, into the page once it passes $180^\circ$, which is the antisymmetry $\mathbf{b}\times\mathbf{a} = -\mathbf{a}\times\mathbf{b}$ seen from the other side.

In [ ]:
from emlib import show
show.animation("m1_fields", "CrossProductArea")

In [ ]:
a = np.array([3.0, 1.0, 0.0]); b = np.array([1.0, 2.0, 0.0])
print("a x b =", np.cross(a, b), "   b x a =", np.cross(b, a), "   a x a =", np.cross(a, a))
a2 = np.array([1.0, 2.0, 3.0]); b2 = np.array([4.0, 5.0, 6.0])
c2 = np.cross(a2, b2)
print("(1,2,3) x (4,5,6) =", c2, "   perpendicular checks:", c2 @ a2, c2 @ b2)
print("unit vectors: x cross y =", np.cross([1, 0, 0], [0, 1, 0]), "  y cross x =", np.cross([0, 1, 0], [1, 0, 0]))
A, B, C = np.array([1.0, 0, 0]), np.array([0, 2.0, 0]), np.array([0, 0, 3.0])
print("triple product a . (b x c) =", A @ np.cross(B, C), "   same as the determinant:", np.linalg.det(np.array([A, B, C])))

In [ ]:
# Worked example 11: triangle area and torque.
P, Qv, Rv = np.array([0.0, 0, 0]), np.array([2.0, 0, 0]), np.array([1.0, 3, 0])
print("triangle area = half the parallelogram:", 0.5 * np.linalg.norm(np.cross(Qv - P, Rv - P)))
for r, F in ((np.array([0.5, 0, 0]), np.array([0, 2.0, 0])), (np.array([0, 0.5, 0]), np.array([0, 2.0, 0])), (np.array([1.0, 2, 0]), np.array([3.0, 1, 0]))):
    print(f"torque of F = {F} applied at r = {r}: r x F = {np.cross(r, F)}")


### 3.4 Position and displacement vectors

Every point in space is labelled by its position vector $\mathbf{r} = (x, y, z)$, the arrow from the origin to the point. The origin is a choice; moving it changes every position vector by the same constant and changes no difference between positions, which is why physics is written in terms of differences.

The most important vector in electrostatics is such a difference: the displacement from a source point $\mathbf{r}'$ (where a charge sits) to a field point $\mathbf{r}$ (where we ask for the field). The prime is the convention used throughout the book and these notebooks: primed coordinates belong to sources and are summed or integrated over, unprimed coordinates belong to the observation point and are held fixed.

$$\mathbf{R} = \mathbf{r} - \mathbf{r}', \qquad |\mathbf{R}| = \sqrt{(x-x')^2 + (y-y')^2 + (z-z')^2}, \qquad \hat{\mathbf{R}} = \mathbf{R}/|\mathbf{R}|.$$

The order matters. $\mathbf{r} - \mathbf{r}'$ points from the source to the field point, which is the direction in which a positive charge pushes. Writing $\mathbf{r}' - \mathbf{r}$ by mistake flips every field.

Coulomb's law in chapter 2 reads $\mathbf{E} = \dfrac{q}{4\pi\epsilon_0}\dfrac{\hat{\mathbf{R}}}{|\mathbf{R}|^2} = \dfrac{q}{4\pi\epsilon_0}\dfrac{\mathbf{R}}{|\mathbf{R}|^3}$. The two forms are the same thing: the unit vector $\hat{\mathbf{R}}$ is $\mathbf{R}$ divided by one length, and the inverse square supplies two more, so the vector $\mathbf{R}$ sits over the cube of its length. The second form avoids computing the unit vector separately and is what the code in later notebooks uses. The constant $1/4\pi\epsilon_0$ is about $9\times10^9$ in SI units; in this primer it is set to 1 so that the geometry stands out.

**Worked example 12.** A charge at $\mathbf{r}' = (-1, 1, 0)$, field point $\mathbf{r} = (2, 1, 0)$. Then $\mathbf{R} = (3, 0, 0)$, $|\mathbf{R}| = 3$, $\hat{\mathbf{R}} = (1, 0, 0)$, and $\mathbf{R}/|\mathbf{R}|^3 = (3, 0, 0)/27 = (1/9, 0, 0)$. The field points away from the charge along $+x$ and its size falls as one over distance squared: at twice the distance, $\mathbf{r} = (5, 1, 0)$, we get $\mathbf{R}/|\mathbf{R}|^3 = (6, 0, 0)/216 = (1/36, 0, 0)$, a quarter as large.

**Worked example 13 (two charges, and what they look like from far away).** Charges $+1$ at $(0, 0.5, 0)$ and $-1$ at $(0, -0.5, 0)$, field point $(1, 0, 0)$. From the first: $\mathbf{R}_1 = (1, -0.5, 0)$, $|\mathbf{R}_1| = \sqrt{1.25} = 1.118$, $|\mathbf{R}_1|^3 = 1.398$, contribution $(+1)(1, -0.5, 0)/1.398 = (0.716, -0.358, 0)$. From the second: $\mathbf{R}_2 = (1, 0.5, 0)$, same length, contribution $(-1)(1, 0.5, 0)/1.398 = (-0.716, -0.358, 0)$. Sum: $(0, -0.716, 0)$. The $x$ parts cancel and the field points along $-y$, from the positive charge round to the negative one.

Now move the field point out to $(10, 0, 0)$. Each $|\mathbf{R}|^3 = (100.25)^{3/2} = 1003.8$, the contributions are $(0.00996, -0.000498, 0)$ and $(-0.00996, -0.000498, 0)$, and the sum is $(0, -0.000996, 0)$, which is $-1/x^3$ to three figures. A single charge would give $1/x^2 = 0.01$ at this distance, ten times larger. The pair, with zero total charge, is a dipole, and its field falls as $1/r^3$ because the two inverse-square fields nearly cancel. If instead both charges were $+1$ the contributions would add to $(0.0199, 0, 0)$, which is $2/x^2$: from far away a cluster with net charge looks like a single charge carrying the total. This is the general pattern: at distances much larger than the size of a charge distribution, the net charge dominates if it is non-zero, and the dipole term takes over if it is not. The exercise below draws the dipole field.


In [ ]:
r_source = np.array([-1.0, 1.0, 0.0])
for r_field in (np.array([2.0, 1.0, 0.0]), np.array([5.0, 1.0, 0.0]), np.array([2.0, 4.0, 0.0])):
    R = r_field - r_source
    print(f"r = {r_field}:  R = {R}  |R| = {np.linalg.norm(R):.3f}  R_hat = {R / np.linalg.norm(R)}  R/|R|^3 = {R / np.linalg.norm(R) ** 3}")

In [ ]:
# Worked example 13: two charges by superposition, near and far.
srcs = np.array([[0.0, 0.5, 0.0], [0.0, -0.5, 0.0]])
for charges, label in (((1.0, -1.0), "dipole (+1, -1)"), ((1.0, 1.0), "pair (+1, +1)")):
    for rf in (np.array([1.0, 0.0, 0.0]), np.array([10.0, 0.0, 0.0])):
        E = sum(q * (rf - s) / np.linalg.norm(rf - s) ** 3 for q, s in zip(charges, srcs))
        print(f"{label:16s} at x = {rf[0]:4.0f}: E = {E}    1/x^2 = {1 / rf[0]**2:.4f}   1/x^3 = {1 / rf[0]**3:.4f}")


### 3.5 Many vectors at once

Physics code rarely handles one vector. It handles thousands: the positions of all the grid points, the field at each of them. The NumPy idiom is an array of shape `(N, 3)`, one vector per row. Lengths come from `np.linalg.norm(..., axis=1)`, row-wise dot products from `np.einsum("ij,ij->i", A, B)` or `np.sum(A * B, axis=1)`, and row-wise cross products from `np.cross`. Broadcasting lets a single vector act on every row.

The cell below computes $\mathbf{R}/|\mathbf{R}|^3$ for one source and many field points in one line, which is already the core of the Coulomb superposition code in module 1.

In [ ]:
points = np.array([[2.0, 1.0, 0.0], [5.0, 1.0, 0.0], [2.0, 4.0, 0.0], [0.0, 0.0, 1.0]])      # (N, 3)
R = points - r_source                                   # broadcasting: the single source is subtracted from every row
dist = np.linalg.norm(R, axis=1)                        # (N,)
coulomb_shape = R / dist[:, None] ** 3                  # divide each row by its own |R|^3
print("distances:", dist)
print("R / |R|^3 per row:\n", coulomb_shape)
print("row-wise dot products of R with itself (= |R|^2):", np.einsum("ij,ij->i", R, R))

### 3.6 Scalar fields (section 1.2)

A function of one variable, $f(x)$, is a curve. A function of position, $f(x, y, z)$, is a scalar field: a number attached to every point in space. Temperature in a room, pressure in the atmosphere, and the electric potential $V$ in chapter 3 are all scalar fields. The word "field" carries no mystery; it means "a function whose input is a position".

Pictures. In two dimensions a scalar field can be drawn as a landscape (height $= f$) or, more usefully, as a contour map: curves along which $f$ is constant, like the height lines on a hiking map. The reading rules are the ones a hiker uses. Closely spaced contours mean steep slopes, because the value changes by one contour step over a short distance. Widely spaced contours mean flat ground. A set of closed loops nested inside each other surrounds a peak or a pit, and you tell which from the labels. Two contours of different values can never cross or touch, since the crossing point would have to carry both values at once. Contours do not end in the interior of the region; they either close on themselves or run out through the edge. In three dimensions the analogues are level surfaces, shells on which $f$ is constant: the equipotential surfaces of chapter 3, which are nested around a point charge like the layers of an onion.

To draw a field we evaluate it on a grid. The grid is every combination of a list of $x$ values with a list of $y$ values, and `np.meshgrid` builds the two arrays that hold the $x$ and $y$ coordinate of every grid point. One convention matters: with `indexing="ij"` the first array index follows $x$ and the second follows $y$, matching how we write $f_{i,j}$ on paper. The default mode swaps the two axes and silently transposes every picture. Every notebook in this repository uses `"ij"`.


In [ ]:
x = np.linspace(-2, 2, 5)
y = np.linspace(0, 1, 3)
X, Y = np.meshgrid(x, y, indexing="ij")
print("x values:", x); print("y values:", y)
print("X (one row per x value, one column per y value):\n", X)
print("Y:\n", Y)
print("grid point [i=3, j=2] sits at (x[3], y[2]) =", (X[3, 2], Y[3, 2]), "=", (x[3], y[2]))

In [ ]:
x = np.linspace(-2, 2, 81); y = np.linspace(-2, 2, 81)
X, Y = np.meshgrid(x, y, indexing="ij")
h = x[1] - x[0]

f = X**2 / 2 + Y**2 - X * Y / 2                     # a tilted bowl
g = np.exp(-((X - 0.8)**2 + Y**2)) - np.exp(-((X + 0.8)**2 + Y**2))   # a hill next to a hollow

fig = plt.figure(figsize=(13, 9))
for k, (F, name) in enumerate(((f, "bowl f"), (g, "hill and hollow g"))):
    ax1 = fig.add_subplot(2, 2, 2 * k + 1, projection="3d")
    ax1.plot_surface(X, Y, F, cmap="viridis", alpha=0.9); ax1.set_title(name + " as a landscape"); ax1.set_xlabel("x"); ax1.set_ylabel("y")
    ax2 = fig.add_subplot(2, 2, 2 * k + 2)
    cs = ax2.contour(X, Y, F, levels=15, cmap="viridis"); ax2.clabel(cs, fontsize=7); ax2.set_aspect("equal"); ax2.set_title(name + " as a contour map")
plt.tight_layout(); plt.show()

Read the contour maps. The bowl has one minimum at the origin, closed contours around it, and contours that crowd together as you go out: the slope increases. The hill-and-hollow field has a maximum near $x = 0.8$ and a minimum near $x = -0.8$, and the contour $g = 0$ runs straight along the $y$ axis between them.

**Worked example 14.** Where is the bowl at height 1? Solve $\tfrac12x^2 + y^2 - \tfrac12xy = 1$. Along the $x$ axis ($y = 0$): $x^2 = 2$, $x = \pm1.414$. Along the $y$ axis ($x = 0$): $y = \pm1$. Along the diagonal $x = y = t$: $\tfrac12t^2 + t^2 - \tfrac12t^2 = t^2 = 1$, $t = \pm1$. So the contour $f = 1$ passes through $(\pm1.414, 0)$, $(0, \pm1)$ and $(\pm1, \pm1)$: an ellipse, tilted, as in the picture.

**Worked example 15.** The hill-and-hollow field is $g = e^{-(x-0.8)^2 - y^2} - e^{-(x+0.8)^2 - y^2}$. Replacing $x$ by $-x$ swaps the two terms, so $g(-x, y) = -g(x, y)$: the field is antisymmetric, and the whole $y$ axis is the contour $g = 0$. Along the $x$ axis: $g(0, 0) = 0$; $g(0.8, 0) = 1 - e^{-2.56} = 1 - 0.077 = 0.923$; $g(-0.8, 0) = -0.923$; $g(2, 0) = e^{-1.44} - e^{-7.84} = 0.237 - 0.000 = 0.237$. The maximum is not exactly at $x = 0.8$, because the hollow pulls the value down more on the side nearer to it; it sits slightly outside, and the code finds where.


In [ ]:
f_func = lambda px, py: px**2 / 2 + py**2 - px * py / 2
for pt in ((1.414, 0), (0, 1), (1, 1), (-1, -1)):
    print(f"f{pt} = {f_func(*pt):.3f}")

In [ ]:
g_func = lambda px, py: np.exp(-((px - 0.8)**2 + py**2)) - np.exp(-((px + 0.8)**2 + py**2))
for pt in ((0, 0), (0.8, 0), (-0.8, 0), (2, 0), (0.5, 0.7), (-0.5, 0.7)):
    print(f"g{pt} = {g_func(*pt):+.3f}")
xx = np.linspace(0, 2, 20001)
print(f"maximum of g along the x axis is at x = {xx[np.argmax(g_func(xx, 0))]:.3f}, slightly beyond 0.8")


### 3.7 Vector fields

A vector field attaches a vector to every point: wind velocity on a weather map, water velocity in a river, and the electric and magnetic fields. On a grid it is stored as one array per component, or as a single array with a trailing axis of length 2 or 3.

Two pictures are in use. A quiver plot draws an arrow at a sample of points; the arrow's direction is the field's direction and its length is the field's magnitude. A stream plot draws field lines, curves that are tangent to the field at every point, which is what you get by dropping a particle into the flow and tracing its path. Field lines show magnitude by how closely they bunch, and direction by arrowheads along them.

For the electric field the field lines obey rules that follow from the physics of chapters 2 and 4, and they are worth knowing before the physics is derived: lines start on positive charges and end on negative charges (or run off to infinity if there is no charge to end on); the number of lines leaving a charge is proportional to the charge; lines never cross, because the field has one direction at each point; and lines bunch together where the field is strong and spread apart where it is weak. One caution about `streamplot`: it spaces its lines for a pleasing picture, not in proportion to the field strength, so the line density in a stream plot is qualitative. The colour of the lines in the plots below carries the magnitude instead.

The cell below draws six reference fields. Look at each picture and answer two questions before reading further: does anything pile up or spread out (that will be divergence), and would a small paddlewheel spin anywhere (that will be curl)? The answers for all six come in notebook 00c.


In [ ]:
x = np.linspace(-2, 2, 21); y = np.linspace(-2, 2, 21)
X, Y = np.meshgrid(x, y, indexing="ij")
reference_fields = {
    "uniform   F = (1, 0)":     (np.ones_like(X), np.zeros_like(X)),
    "radial    F = (x, y)":     (X, Y),
    "rotation  F = (-y, x)":    (-Y, X),
    "saddle    F = (x, -y)":    (X, -Y),
    "shear     F = (y, 0)":     (Y, np.zeros_like(X)),
    "sink+swirl F = (-x-y, x-y)": (-X - Y, X - Y),
}
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (name, (Fx, Fy)) in zip(axes.flat, reference_fields.items()):
    ax.quiver(X, Y, Fx, Fy, pivot="mid")
    ax.set_aspect("equal"); ax.set_title(name)
plt.show()

Two of these deserve a comment now. The shear field $(y, 0)$ has all its arrows parallel, yet a paddlewheel placed in it spins: the flow is faster on one side of the wheel than the other. Parallel arrows do not mean no rotation. The sink-with-swirl field is the radial field pointed inward plus the rotation field; it does both things at once. Divergence and curl are independent properties.

Field lines are often the better picture for electric and magnetic fields. The cell draws the same fields as stream plots. Note that `streamplot` expects arrays indexed as `[y, x]`, so the `"ij"` arrays are transposed on the way in. That is the one place in these notebooks where the transpose appears, and it is deliberate.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (name, (Fx, Fy)) in zip(axes.flat, reference_fields.items()):
    speed = np.hypot(Fx, Fy)
    ax.streamplot(x, y, Fx.T, Fy.T, color=speed.T, cmap="viridis", density=1.2)
    ax.set_aspect("equal"); ax.set_title(name + "  (field lines, colour = speed)")
plt.show()

### 3.8 The field of a point charge as a vector field

Combine sections 3.4 and 3.7: the Coulomb field of a charge at the origin is $\mathbf{E} \propto \mathbf{r}/|\mathbf{r}|^3$. Its direction is radial and its magnitude is $1/r^2$. On a quiver plot the arrows near the charge are enormous and the far ones invisible, because the magnitude changes by a factor of a hundred across a plot that spans a factor of ten in distance. It is usual to draw unit arrows and encode the magnitude as colour on a logarithmic scale. The code below does that, and the stream plot shows the familiar radial field lines, which start on the charge and run off to infinity: a positive charge is a source of field lines and, by the rules above, nothing in the picture can end them.

**Worked example 16.** At $\mathbf{r} = (1, 0, 0)$ the field is $(1, 0, 0)/1 = (1, 0, 0)$ in units of $q/4\pi\epsilon_0$. At $(0, 2, 0)$ it is $(0, 2, 0)/8 = (0, 0.25, 0)$: pointing along $+y$, a quarter as strong, as $1/r^2$ demands. At $(1, 1, 0)$: $|\mathbf{r}| = \sqrt2$, $|\mathbf{r}|^3 = 2\sqrt2 = 2.828$, so $\mathbf{E} = (0.354, 0.354, 0)$ with magnitude $0.5 = 1/r^2$.


In [ ]:
x = np.linspace(-2, 2, 41); y = np.linspace(-2, 2, 41)
X, Y = np.meshgrid(x, y, indexing="ij")
Rmag = np.hypot(X, Y)
with np.errstate(divide="ignore", invalid="ignore"):
    Ex, Ey = X / Rmag**3, Y / Rmag**3
Emag = np.hypot(Ex, Ey)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
st = 2
axes[0].quiver(X[::st, ::st], Y[::st, ::st], (Ex / Emag)[::st, ::st], (Ey / Emag)[::st, ::st], np.log10(Emag)[::st, ::st], cmap="plasma", pivot="mid")
axes[0].set_aspect("equal"); axes[0].set_title("point charge: unit arrows, colour = log10 |E|")
axes[1].streamplot(x, y, Ex.T, Ey.T, color=np.log10(Emag).T, cmap="plasma", density=1.3)
axes[1].set_aspect("equal"); axes[1].set_title("field lines radiate from the charge")
plt.show()
for p in ((1.0, 0.0), (0.0, 2.0), (1.0, 1.0)):
    r = np.array(p); print(f"E at {p} = {r / np.linalg.norm(r) ** 3}   magnitude {1 / np.linalg.norm(r) ** 2:.3f}")

### Exercise: a field from a formula

Replace the body of `dipole_field` so that it returns the 2D field of two charges, $+1$ at $(0, 0.5)$ and $-1$ at $(0, -0.5)$, as the sum of two Coulomb terms $\mathbf{R}/|\mathbf{R}|^3$. The check evaluates it on the $x$ axis far away, where the field of a dipole points along $-y$ and falls as $1/r^3$ (for this dipole, $E_y \approx -1/x^3$ at large $x$).

In [ ]:
def dipole_field(X, Y):
    """Return (Ex, Ey) of a +1 charge at (0, 0.5) and a -1 charge at (0, -0.5), in units of q/(4 pi eps0)."""
    Ex = np.zeros_like(X); Ey = np.zeros_like(Y)
    for q, (sx, sy) in ((1.0, (0.0, 0.5)), (-1.0, (0.0, -0.5))):               # replace this loop with your own version
        Rx, Ry = X - sx, Y - sy
        d3 = np.hypot(Rx, Ry) ** 3
        Ex = Ex + q * Rx / d3; Ey = Ey + q * Ry / d3
    return Ex, Ey


xs = np.array([5.0, 10.0, 20.0])
Ex_far, Ey_far = dipole_field(xs, np.zeros_like(xs))
print("on the x axis:  E_y * x^3 =", Ey_far * xs**3, "  (approaches -1)   E_x =", Ex_far)
assert np.allclose(Ey_far * xs**3, -1.0, rtol=0.02) and np.allclose(Ex_far, 0.0)
Ex_d, Ey_d = dipole_field(X, Y)
plt.streamplot(x, y, Ex_d.T, Ey_d.T, color="k", density=1.3, linewidth=0.7)
plt.plot([0, 0], [0.5, -0.5], "o", color="C3"); plt.gca().set_aspect("equal"); plt.title("your dipole field"); plt.show()

## 4. Verify

Three identities that every vector calculation relies on, checked on random vectors so that they cannot pass by accident:

1. $|\mathbf{a}\times\mathbf{b}|^2 + (\mathbf{a}\cdot\mathbf{b})^2 = |\mathbf{a}|^2|\mathbf{b}|^2$ (because $\sin^2 + \cos^2 = 1$).
2. $\mathbf{a}\times(\mathbf{b}\times\mathbf{c}) = \mathbf{b}(\mathbf{a}\cdot\mathbf{c}) - \mathbf{c}(\mathbf{a}\cdot\mathbf{b})$, the "BAC minus CAB" rule, used when the magnetic vector potential appears in chapter 11.
3. $\mathbf{a}\cdot(\mathbf{b}\times\mathbf{c}) = \mathbf{b}\cdot(\mathbf{c}\times\mathbf{a}) = \mathbf{c}\cdot(\mathbf{a}\times\mathbf{b})$, cyclic symmetry of the triple product.

In [ ]:
rng = np.random.default_rng(1)
A, B, C = rng.normal(size=(3, 3))
lhs1 = np.linalg.norm(np.cross(A, B))**2 + (A @ B)**2
rhs1 = (A @ A) * (B @ B)
lhs2 = np.cross(A, np.cross(B, C)); rhs2 = B * (A @ C) - C * (A @ B)
t1, t2, t3 = A @ np.cross(B, C), B @ np.cross(C, A), C @ np.cross(A, B)
print(f"identity 1: {lhs1:.6f} = {rhs1:.6f}")
print(f"identity 2: {lhs2} = {rhs2}")
print(f"identity 3: {t1:.6f} = {t2:.6f} = {t3:.6f}")
assert np.isclose(lhs1, rhs1) and np.allclose(lhs2, rhs2) and np.isclose(t1, t2) and np.isclose(t2, t3)

## 5. Explore and rebuild

Explore. Build the field of four charges at the corners of a square, alternating in sign (a quadrupole). Draw its field lines. Then evaluate its magnitude along a diagonal at distances 5, 10, 20, 40 and find how fast it falls off: fit the exponent in $|\mathbf{E}| \propto r^{-n}$. Compare with the monopole ($n = 2$) and dipole ($n = 3$) and explain the pattern in terms of cancellation, using worked example 13 as the model.

Rebuild. Close this notebook. On paper, write the three faces of the dot product and the component formula for the cross product, then compute worked examples 4 and 9 again by hand. In a blank editor, write a function that takes a source position, a charge, and an `(N, 3)` array of field points and returns the Coulomb field at every point in one broadcast expression.


## 6. Key results

| result | formula | holds when |
|---|---|---|
| length | $\lvert\mathbf{a}\rvert = \sqrt{a_x^2 + a_y^2 + a_z^2} = \sqrt{\mathbf{a}\cdot\mathbf{a}}$ | always; independent of the axes |
| unit vector | $\hat{\mathbf{a}} = \mathbf{a}/\lvert\mathbf{a}\rvert$ | $\mathbf{a} \ne 0$ |
| addition, scaling | componentwise; head to tail | always |
| components in new axes | $a_x' = \mathbf{a}\cdot\hat{\mathbf{x}}'$, etc. | axes are unit and perpendicular |
| dot product | $\mathbf{a}\cdot\mathbf{b} = a_xb_x + a_yb_y + a_zb_z = \lvert\mathbf{a}\rvert\lvert\mathbf{b}\rvert\cos\theta$ | always |
| projection | along $\mathbf{b}$: $(\mathbf{a}\cdot\hat{\mathbf{b}})\hat{\mathbf{b}}$; perpendicular remainder: $\mathbf{a} - (\mathbf{a}\cdot\hat{\mathbf{b}})\hat{\mathbf{b}}$ | $\mathbf{b} \ne 0$ |
| bound | $\lvert\mathbf{a}\cdot\mathbf{b}\rvert \le \lvert\mathbf{a}\rvert\lvert\mathbf{b}\rvert$ | equality only if parallel |
| law of cosines | $\lvert\mathbf{a} - \mathbf{b}\rvert^2 = \lvert\mathbf{a}\rvert^2 + \lvert\mathbf{b}\rvert^2 - 2\mathbf{a}\cdot\mathbf{b}$ | always |
| flux through a flat surface | $\mathbf{E}\cdot\hat{\mathbf{n}}\,A = \lvert\mathbf{E}\rvert A\cos\theta$ | $\mathbf{E}$ uniform over the surface |
| cross product | $(a_yb_z - a_zb_y,\ a_zb_x - a_xb_z,\ a_xb_y - a_yb_x)$; length $\lvert\mathbf{a}\rvert\lvert\mathbf{b}\rvert\sin\theta$; right-hand rule | always |
| antisymmetry | $\mathbf{b}\times\mathbf{a} = -\mathbf{a}\times\mathbf{b}$, $\mathbf{a}\times\mathbf{a} = 0$ | always |
| area | parallelogram $\lvert\mathbf{a}\times\mathbf{b}\rvert$, triangle half of that | sides from one vertex |
| torque | $\boldsymbol{\tau} = \mathbf{r}\times\mathbf{F}$ | $\mathbf{r}$ from the pivot to the point of application |
| volume | $\mathbf{a}\cdot(\mathbf{b}\times\mathbf{c})$, signed | always |
| vector triple product | $\mathbf{a}\times(\mathbf{b}\times\mathbf{c}) = \mathbf{b}(\mathbf{a}\cdot\mathbf{c}) - \mathbf{c}(\mathbf{a}\cdot\mathbf{b})$ | always |
| source to field point | $\mathbf{R} = \mathbf{r} - \mathbf{r}'$ (unprimed: observation; primed: source) | always |
| Coulomb field | $\mathbf{E} = \dfrac{q}{4\pi\epsilon_0}\dfrac{\mathbf{R}}{\lvert\mathbf{R}\rvert^3}$ | point charge at rest |
| superposition | fields and forces from several charges add as vectors | always |
| far field | net charge $\ne 0$: $\lvert\mathbf{E}\rvert \propto 1/r^2$; net charge $= 0$ (dipole): $\propto 1/r^3$ | $r \gg$ size of the distribution |
| grid convention | `np.meshgrid(x, y, indexing="ij")`: `F[i, j]` sits at `(x[i], y[j])` | every notebook here |
| contour rules | spacing = steepness, nested loops = extremum, no crossings | any scalar field |
| field-line rules | start on $+$, end on $-$ or at infinity, never cross, bunch where strong | electrostatic field |


## 7. Warm-up answers

1. A vector is an arrow with a length and a direction. Its components are the arrow's extents along three chosen axes. Choose different axes and the same arrow gets different components; its length and its angles with other vectors do not change (worked example 2).
2. $\mathbf{a}\cdot\mathbf{b}$ is a number, $|\mathbf{a}||\mathbf{b}|\cos\theta$: how much of one vector lies along the other. $\mathbf{a}\times\mathbf{b}$ is a vector perpendicular to both, of length $|\mathbf{a}||\mathbf{b}|\sin\theta$: the area they span and the axis of the turn from $\mathbf{a}$ to $\mathbf{b}$.
3. $(3, 4, 0)/5 = (0.6, 0.8, 0)$.
4. Scalar field: temperature in a room, drawn as a contour map or a landscape. Vector field: wind velocity, drawn as arrows or as streamlines.
5. $\mathbf{R} = \mathbf{r} - \mathbf{r}'$, from the source position $\mathbf{r}'$ to the observation point $\mathbf{r}$. The field is $\propto \mathbf{R}/|\mathbf{R}|^3$.
6. A contour is the set of points where the field has one particular value. At a crossing point the field would have to equal two different values at once, which a function cannot do.
7. The field is strong there. Field lines bunch where the magnitude is large and spread where it is small, so line density is a picture of magnitude (qualitatively in a stream plot, exactly in a field-line drawing that follows the flux rules of chapter 4).
